# 🎯 Custom Object Detection Model Prep (YOLOX-Tiny)

This notebook converts a pretrained PyTorch YOLOX-Tiny model into an INT8 quantized TFLite model, ready for compilation.

### Pipeline:
1. PyTorch (`.pth`) -> ONNX
2. ONNX -> TFLite (FP32)
3. TFLite (FP32) -> TFLite (INT8) using Post-Training Quantization (PTQ) with COCO calibration images.

*Once the `.tflite` model is generated, use `convert_model.py` to compile it to C code.*

In [ ]:
# Install dependencies
!pip install torch onnx onnxsim onnx2tf tensorflow opencv-python tqdm

## 1. Setup & Configuration

In [ ]:
import os
import pathlib
import shutil
import urllib.request
import zipfile
import numpy as np
import cv2
import torch
from torch import nn
import tensorflow as tf
import onnx2tf
from tqdm import tqdm

# Configuration
NUM_CALIB_SAMPLES = 200
INPUT_H, INPUT_W = 224, 224
NUM_CLASSES = 80
PAD_VALUE = 114

# Paths
WORKSPACE_DIR = pathlib.Path(os.getcwd())
MODEL_DIR = WORKSPACE_DIR / "model_output"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

PTH_PATH = MODEL_DIR / "yolox_tiny.pth"
ONNX_PATH = MODEL_DIR / "yolox_tiny_224.onnx"
FP32_PATH = MODEL_DIR / "yolox_tiny_224_FP32.tflite"
INT8_PATH = MODEL_DIR / "yolox_tiny_224_INT8.tflite"
CALIB_DIR = MODEL_DIR / "val2017"

print(f"Workspace: {WORKSPACE_DIR}")
print(f"Output directory: {MODEL_DIR}")

## 2. Locate YOLOX-Tiny PyTorch Utils
We need to load the model definition from the existing repository.

In [ ]:
import sys
# Automatically find the vision repo to import yolox_model utils
_repo_root = WORKSPACE_DIR.parent if WORKSPACE_DIR.name == "tutorials" else WORKSPACE_DIR
if not (_repo_root / "vision").exists():
    for _p in WORKSPACE_DIR.parents:
        if (_p / "vision").exists():
            _repo_root = _p
            break

_utils_dir = str(_repo_root / "vision" / "object_detection" / "yolox_tiny" / "python")
if _utils_dir not in sys.path:
    sys.path.insert(0, _utils_dir)

from utils.yolox_model import build_yolox_tiny, replace_module, SiLU
print("YOLOX utilities successfully loaded.")

## 3. Download Model & Convert to ONNX

In [ ]:
def download_file(url, dest):
    if not dest.exists():
        print(f"Downloading {url}...")
        urllib.request.urlretrieve(url, str(dest))
        print("Download complete.")

# Download PyTorch checkpoint
PTH_URL = "https://github.com/Megvii-BaseDetection/YOLOX/releases/download/0.1.1rc0/yolox_tiny.pth"
download_file(PTH_URL, PTH_PATH)

# Convert to ONNX
if not ONNX_PATH.exists():
    print(f"Exporting PyTorch to ONNX ({INPUT_H}x{INPUT_W})...")
    model = build_yolox_tiny(num_classes=NUM_CLASSES)
    ckpt = torch.load(str(PTH_PATH), map_location="cpu")
    if "model" in ckpt:
        ckpt = ckpt["model"]
    model.load_state_dict(ckpt)
    model.eval()
    model = replace_module(model, nn.SiLU, SiLU)
    model.head.decode_in_inference = False

    dummy = torch.randn(1, 3, INPUT_H, INPUT_W)
    torch.onnx.export(model, dummy, str(ONNX_PATH), input_names=["images"], output_names=["output"], opset_version=18)

    # Simplify
    try:
        import onnx
        from onnxsim import simplify
        m = onnx.load(str(ONNX_PATH))
        m_simp, ok = simplify(m)
        if ok:
            onnx.save(m_simp, str(ONNX_PATH))
            print("ONNX simplified with onnxsim")
    except Exception as e:
        print(f"Could not simplify: {e}")

print(f"ONNX Model Ready: {ONNX_PATH}")

## 4. Convert ONNX to TFLite (FP32)

In [ ]:
if not FP32_PATH.exists():
    print("Converting ONNX to TFLite FP32 using onnx2tf...")
    sm_fp32 = str(MODEL_DIR / "saved_model_fp32")
    onnx2tf.convert(
        input_onnx_file_path=str(ONNX_PATH),
        output_folder_path=sm_fp32,
        copy_onnx_input_output_names_to_tflite=True,
        non_verbose=True,
    )

    tflite_files = list(pathlib.Path(sm_fp32).rglob("*.tflite"))
    target_file = next((f for f in tflite_files if "float32" in f.name.lower()), tflite_files[0])
    shutil.copy2(target_file, FP32_PATH)
    shutil.rmtree(sm_fp32, ignore_errors=True)

print(f"TFLite FP32 Ready: {FP32_PATH}")

## 5. Calibrate & Quantize to INT8

In [ ]:
def preprocess_nhwc(image_path, input_size):
    img = cv2.imread(str(image_path))
    if img is None: return None
    ih, iw = img.shape[:2]
    scale = min(input_size / iw, input_size / ih)
    nw, nh = int(iw * scale), int(ih * scale)
    resized = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_LINEAR)
    padded = np.full((input_size, input_size, 3), PAD_VALUE, dtype=np.uint8)
    pw, ph = (input_size - nw) // 2, (input_size - nh) // 2
    padded[ph:ph + nh, pw:pw + nw] = resized
    return padded.astype(np.float32)[np.newaxis, ...]  # (1,H,W,3)

if not INT8_PATH.exists():
    # Download Calibration Data (COCO val2017)
    COCO_VAL_URL = "http://images.cocodataset.org/zips/val2017.zip"
    ZIP_PATH = MODEL_DIR / "val2017.zip"
    if not CALIB_DIR.exists():
        download_file(COCO_VAL_URL, ZIP_PATH)
        print("Extracting COCO val2017...")
        with zipfile.ZipFile(ZIP_PATH, "r") as zf:
            zf.extractall(MODEL_DIR)

    images = list(CALIB_DIR.glob("*.jpg"))[:NUM_CALIB_SAMPLES]
    print(f"Using {len(images)} images for INT8 calibration.")

    sm_int8 = str(MODEL_DIR / "saved_model_int8")
    onnx2tf.convert(
        input_onnx_file_path=str(ONNX_PATH),
        output_folder_path=sm_int8,
        copy_onnx_input_output_names_to_tflite=True,
        non_verbose=True,
    )

    def representative_dataset():
        for path in images:
            blob = preprocess_nhwc(path, INPUT_H)
            if blob is not None:
                yield [blob]

    print("Running TFLite INT8 quantization...")
    converter = tf.lite.TFLiteConverter.from_saved_model(sm_int8)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.representative_dataset = representative_dataset
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type = tf.int8
    converter.inference_output_type = tf.int8

    tflite_model = converter.convert()
    with open(INT8_PATH, "wb") as f:
        f.write(tflite_model)

    shutil.rmtree(sm_int8, ignore_errors=True)

print(f"TFLite INT8 Ready: {INT8_PATH}")

## ✅ Finished!

Your INT8 `.tflite` model is now ready.
**Next step:** Run `convert_model.py` to compile this `.tflite` model into C-source arrays for the NPU!